# Wasserstein Ascend Descend
### (C. and N.) Garcia Trillos

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys
import numpy as np
from Robust_nn.WAD import WAD2scale
import matplotlib.pyplot as plt
from utils.utils import read_vision_dataset
from utils.convnet import ConvNet
from utils.convnet_silu import ConvNetSiLU
import os
import torch
import gc
import torch.nn as nn
from torch.optim.swa_utils import AveragedModel, SWALR


**Read the DataLoader**

In [3]:
trainloader, testloader = read_vision_dataset('../data', dataset='MNIST',batch_size = 512)

In [13]:
# Create some networks

n_nets = 5
net_lst = [ConvNet() for i in range(n_nets)]
avg_nets =[ AveragedModel(model) for model in net_lst]

adv_net = WAD2scale(net_list = net_lst, avg_nets = avg_nets, 
                    trainloader=trainloader, testloader = testloader, 
                    device = None , criterion= nn.CrossEntropyLoss(), 
                    scale_factor=5, num_adverse=1,
                    kappa = { 'param': 0.2, 'adv': 0.2   },
                    max_batches= 2)

In [15]:
adv_net._sample_no_rep(torch.ones(10)/10,15)

(tensor([0.1000, 0.1000, 0.1000, 0.1000, 0.1000, 0.1000, 0.1000, 0.1000, 0.1000,
         0.1000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000]),
 tensor([0., 1., 2., 3., 4., 5., 6., 7., 8., 9., 0., 0., 0., 0., 0.],
        dtype=torch.float64))

In [10]:

adv_net.set_optimizer()
adv_net.train(epochs = 10)


Epoch: 0
tensor([1.])
tensor([0.2000, 0.2000, 0.2000, 0.2000, 0.2000])
0|1|2|
Epoch: 1
tensor([1.])
tensor([0.1915, 0.1934, 0.2056, 0.1957, 0.2139], grad_fn=<DivBackward0>)
0|1|2|
Epoch: 2
tensor([1.])
tensor([0.1823, 0.1854, 0.2137, 0.1911, 0.2275], grad_fn=<DivBackward0>)
0|1|2|
Epoch: 3
tensor([1.])
tensor([0.1740, 0.1768, 0.2201, 0.1866, 0.2425], grad_fn=<DivBackward0>)
0|1|2|
Epoch: 4
tensor([1.])
tensor([0.1640, 0.1694, 0.2273, 0.1838, 0.2555], grad_fn=<DivBackward0>)
0|1|2|
Epoch: 5
tensor([1.])
tensor([0.1568, 0.1620, 0.2311, 0.1806, 0.2696], grad_fn=<DivBackward0>)
0|1|2|
Epoch: 6
tensor([1.])
tensor([0.1500, 0.1563, 0.2358, 0.1770, 0.2809], grad_fn=<DivBackward0>)
0|1|2|
Epoch: 7
tensor([1.])
tensor([0.1443, 0.1505, 0.2409, 0.1727, 0.2917], grad_fn=<DivBackward0>)
0|1|2|
Epoch: 8
tensor([1.])
tensor([0.1388, 0.1447, 0.2455, 0.1701, 0.3009], grad_fn=<DivBackward0>)
0|1|2|
Epoch: 9
tensor([1.])
tensor([0.1331, 0.1396, 0.2506, 0.1672, 0.3096], grad_fn=<DivBackward0>)
0|1|2|

In [11]:
adv_net.save_model('Test1_kappa0p2')

Saving...


In [19]:
adv_net.set_optimizer()
adv_net.import_model('Test1_kappa0p2')

Loading...
done.


In [20]:
adv_net.test(5)

RuntimeError: Input type (torch.cuda.FloatTensor) and weight type (torch.FloatTensor) should be the same

**Creating and comparing results**

[tensor([0.1745, 0.3892, 0.6346, 0.0489, 0.0594]), tensor([0.0834, 0.4240, 0.5160, 0.3686, 0.5654]), tensor([0.6288, 0.8525, 0.3783, 0.9661, 0.8952])]
tensor([0.8867, 1.6657, 1.5288, 1.3836, 1.5201])


In [ ]:
# options = {'only o1':(True,False, False), 'only o2':(False, True, False), 'both': (True,True, False), 'none':(False,False,False), 'modO2':(False,True,True) } 
options = {'only o1':(True,False, False), 'both': (True,True, True), 'none':(False,False,False) } 
rvec = [2,4,np.inf]
deltav = [0.2]
mdict = {}
basepath = os.curdir
mpath = os.path.join(basepath,'models')

for i,(k,(o1,o2, mod_o2)) in enumerate(options.items()):
  auxd = {}
  for r in rvec:
    rstr = str(r)
    print('\n*******\n ** Case '+k+', r=',r,'\n ******')
    torch.manual_seed(0)
    np.random.seed(0)
    auxd[rstr] = {}
    network = ConvNet()
    net_RegTrin = OrdTwoL(network, trainloader, testloader,  device='cuda', delta =0.2, r= r, o2=o2, o1=o1, mod_o2=mod_o2) #'cuda'
    net_RegTrin.set_optimizer(optim_alg='Adam', args={'lr':1e-4})
    net_RegTrin.train(epochs=5, delta=deltav)
    auxd[rstr]['train_loss'] = net_RegTrin.train_loss.copy()
    auxd[rstr]['train_acc'] = net_RegTrin.train_acc.copy()
    auxd[rstr]['train_reg'] = net_RegTrin.train_reg.copy()
    auxd[rstr]['test_acc_adv'] = net_RegTrin.test_acc_adv.copy()
    auxd[rstr]['test_acc_clean'] = net_RegTrin.test_acc_clean.copy()
    auxd[rstr]['train_times'] = net_RegTrin.train_times.copy()
    torch.save(network.state_dict(), os.path.join(mpath, k+'_r_'+str(r)+'.pth' ))
    del(network)
    gc.collect()
    torch.cuda.empty_cache()
  mdict[k] = auxd
  with open('tests_all.txt','w') as data: 
      data.write(str(mdict))